# Notebook 52: Day Trading Optimization - Tight Trail Stops

**Finding from Notebook 51:** 15% trail on 4H gives 8 trades/year

**Goal:** Test even tighter stops (5-15%) to maximize trade frequency while maintaining edge.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"

## 1. Load and Prepare Data

In [ ]:
# Load hourly data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    sopr_lth = pd.read_parquet(HOURLY_DIR / "sopr_lth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["sopr_lth"] = sopr_lth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    
    return df

df_h1 = load_hourly()
print(f"Loaded {len(df_h1):,} hourly bars")

In [ ]:
def resample_to_timeframe(df_h1, timeframe):
    df = pd.DataFrame()
    df["price"] = df_h1["price"].resample(timeframe).last()
    df["sopr"] = df_h1["sopr"].resample(timeframe).mean()
    df["sopr_sth"] = df_h1["sopr_sth"].resample(timeframe).mean()
    df["sopr_lth"] = df_h1["sopr_lth"].resample(timeframe).mean()
    df["realized_loss"] = df_h1["realized_loss"].resample(timeframe).sum()
    return df.dropna()

def add_zscore(df, window_periods):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window_periods, min_periods=window_periods//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window_periods, min_periods=window_periods//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

# Create timeframes
df_1h = add_zscore(df_h1, 365 * 24)
df_4h = add_zscore(resample_to_timeframe(df_h1, "4h"), 365 * 6)
df_8h = add_zscore(resample_to_timeframe(df_h1, "8h"), 365 * 3)
df_12h = add_zscore(resample_to_timeframe(df_h1, "12h"), 365 * 2)

# Filter to backtest period
START_DATE = "2019-01-01"
df_1h = df_1h[df_1h.index >= START_DATE].dropna()
df_4h = df_4h[df_4h.index >= START_DATE].dropna()
df_8h = df_8h[df_8h.index >= START_DATE].dropna()
df_12h = df_12h[df_12h.index >= START_DATE].dropna()

years = (df_4h.index.max() - df_4h.index.min()).days / 365.25
print(f"Backtest: {years:.2f} years")
print(f"1H: {len(df_1h):,} | 4H: {len(df_4h):,} | 8H: {len(df_8h):,} | 12H: {len(df_12h):,}")

In [ ]:
def get_metrics(pf, years):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    trades_per_year = len(trades) / years if years > 0 else 0
    
    if "Entry Timestamp" in trades.columns and "Exit Timestamp" in trades.columns:
        durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
        avg_duration_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    else:
        avg_duration_days = 0
    
    winning_trades = trades[trades["PnL"] > 0]
    losing_trades = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100 if years > 0 else 0,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_per_year": trades_per_year,
        "avg_duration_days": avg_duration_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(winning_trades["PnL"].sum() / losing_trades["PnL"].sum()) if len(losing_trades) > 0 else np.inf,
        "avg_win_pct": winning_trades["Return"].mean() * 100 if len(winning_trades) > 0 else 0,
        "avg_loss_pct": losing_trades["Return"].mean() * 100 if len(losing_trades) > 0 else 0,
    }

## 2. Fine-Grained Trail Stop Test (4H)

In [ ]:
def run_strat002(df, freq, trail_pct=0.30):
    cond = (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None
    
    pf = vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail_pct,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )
    
    return pf

# Fine-grained trail stops for 4H
trail_stops_fine = [0.05, 0.07, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]

print("="*140)
print("4H TIMEFRAME: FINE-GRAINED TRAIL STOP OPTIMIZATION")
print("="*140)
print(f"\n{'Trail':>8} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10} {'AvgWin':>10} {'AvgLoss':>10} {'PF':>8}")
print("-"*140)

results_4h = {}

for trail in trail_stops_fine:
    pf = run_strat002(df_4h, "4h", trail_pct=trail)
    if pf is not None:
        m = get_metrics(pf, years)
        results_4h[trail] = {"metrics": m, "pf": pf}
        print(f"{trail*100:>7.0f}% {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}% {m['avg_win_pct']:>+9.1f}% {m['avg_loss_pct']:>+9.1f}% {m['profit_factor']:>8.2f}")

## 3. Fine-Grained Trail Stop Test (8H)

In [ ]:
print("\n" + "="*140)
print("8H TIMEFRAME: FINE-GRAINED TRAIL STOP OPTIMIZATION")
print("="*140)
print(f"\n{'Trail':>8} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10} {'AvgWin':>10} {'AvgLoss':>10} {'PF':>8}")
print("-"*140)

results_8h = {}

for trail in trail_stops_fine:
    pf = run_strat002(df_8h, "8h", trail_pct=trail)
    if pf is not None:
        m = get_metrics(pf, years)
        results_8h[trail] = {"metrics": m, "pf": pf}
        print(f"{trail*100:>7.0f}% {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}% {m['avg_win_pct']:>+9.1f}% {m['avg_loss_pct']:>+9.1f}% {m['profit_factor']:>8.2f}")

## 4. Fine-Grained Trail Stop Test (1H)

In [ ]:
print("\n" + "="*140)
print("1H TIMEFRAME: FINE-GRAINED TRAIL STOP OPTIMIZATION")
print("="*140)
print(f"\n{'Trail':>8} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10} {'AvgWin':>10} {'AvgLoss':>10} {'PF':>8}")
print("-"*140)

results_1h = {}

for trail in trail_stops_fine:
    pf = run_strat002(df_1h, "1h", trail_pct=trail)
    if pf is not None:
        m = get_metrics(pf, years)
        results_1h[trail] = {"metrics": m, "pf": pf}
        print(f"{trail*100:>7.0f}% {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}% {m['avg_win_pct']:>+9.1f}% {m['avg_loss_pct']:>+9.1f}% {m['profit_factor']:>8.2f}")

## 5. Trades Per Year vs Return Chart

In [ ]:
# Plot trades/year vs return for all timeframes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Trades per year vs Trail Stop
ax1 = axes[0]
for name, results, color in [("1H", results_1h, "blue"), ("4H", results_4h, "green"), ("8H", results_8h, "red")]:
    trails = sorted(results.keys())
    trades_per_yr = [results[t]["metrics"]["trades_per_year"] for t in trails]
    ax1.plot([t*100 for t in trails], trades_per_yr, marker='o', label=name, color=color)

ax1.set_xlabel("Trail Stop (%)")
ax1.set_ylabel("Trades per Year")
ax1.set_title("Trade Frequency vs Trail Stop")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Return vs Trades per year
ax2 = axes[1]
for name, results, color in [("1H", results_1h, "blue"), ("4H", results_4h, "green"), ("8H", results_8h, "red")]:
    trades_per_yr = [results[t]["metrics"]["trades_per_year"] for t in results]
    returns = [results[t]["metrics"]["return"] for t in results]
    ax2.scatter(trades_per_yr, returns, label=name, color=color, s=100)
    
    # Annotate with trail %
    for t in results:
        ax2.annotate(f"{t*100:.0f}%", 
                     (results[t]["metrics"]["trades_per_year"], results[t]["metrics"]["return"]),
                     fontsize=8, alpha=0.7)

ax2.set_xlabel("Trades per Year")
ax2.set_ylabel("Total Return (%)")
ax2.set_title("Return vs Trade Frequency")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Income Analysis

In [ ]:
print("\n" + "="*120)
print("INCOME ANALYSIS: $100K CAPITAL")
print("="*120)

capital = 100000

# Best options for each timeframe
print(f"\n{'TF':<6} {'Trail':>8} {'Tr/Yr':>8} {'CAGR':>10} {'Annual$':>12} {'$/Trade':>12} {'WinRate':>10} {'AvgHold':>10}")
print("-"*100)

# Find best trade frequency options (>5 trades/year)
for name, results in [("1H", results_1h), ("4H", results_4h), ("8H", results_8h)]:
    for trail in sorted(results.keys()):
        m = results[trail]["metrics"]
        if m["trades_per_year"] >= 5:  # At least 5 trades/year
            annual_profit = capital * (m['cagr'] / 100)
            profit_per_trade = annual_profit / m['trades_per_year'] if m['trades_per_year'] > 0 else 0
            print(f"{name:<6} {trail*100:>7.0f}% {m['trades_per_year']:>8.1f} {m['cagr']:>+9.1f}% ${annual_profit:>11,.0f} ${profit_per_trade:>11,.0f} {m['win_rate']:>9.0f}% {m['avg_duration_days']:>9.1f}d")

## 7. Best Configurations Summary

In [ ]:
print("\n" + "="*120)
print("BEST CONFIGURATIONS FOR DIFFERENT GOALS")
print("="*120)

all_configs = []
for name, results in [("1H", results_1h), ("4H", results_4h), ("8H", results_8h)]:
    for trail in results:
        m = results[trail]["metrics"]
        all_configs.append({
            "tf": name,
            "trail": trail,
            **m
        })

df_configs = pd.DataFrame(all_configs)

print("\n📈 BEST TOTAL RETURN:")
best_return = df_configs.loc[df_configs["return"].idxmax()]
print(f"   {best_return['tf']} @ {best_return['trail']*100:.0f}% trail: {best_return['return']:+,.0f}% ({best_return['trades_per_year']:.1f} trades/yr)")

print("\n📊 BEST SHARPE (Risk-Adjusted):")
best_sharpe = df_configs.loc[df_configs["sharpe"].idxmax()]
print(f"   {best_sharpe['tf']} @ {best_sharpe['trail']*100:.0f}% trail: Sharpe {best_sharpe['sharpe']:.2f} ({best_sharpe['return']:+,.0f}%)")

print("\n🔄 MOST TRADES (>5/yr with positive return):")
active = df_configs[(df_configs["trades_per_year"] >= 5) & (df_configs["return"] > 0)]
if len(active) > 0:
    most_trades = active.loc[active["trades_per_year"].idxmax()]
    print(f"   {most_trades['tf']} @ {most_trades['trail']*100:.0f}% trail: {most_trades['trades_per_year']:.1f} trades/yr ({most_trades['return']:+,.0f}%)")

print("\n💰 BEST FOR INCOME (>5 trades/yr, highest CAGR):")
if len(active) > 0:
    best_income = active.loc[active["cagr"].idxmax()]
    annual = 100000 * (best_income['cagr'] / 100)
    print(f"   {best_income['tf']} @ {best_income['trail']*100:.0f}% trail: {best_income['cagr']:+.1f}% CAGR = ${annual:,.0f}/yr on $100K")
    print(f"   {best_income['trades_per_year']:.1f} trades/yr, {best_income['avg_duration_days']:.0f} day avg hold, {best_income['win_rate']:.0f}% win rate")

## 8. Show Actual Trades for Best Income Config

In [ ]:
# Show trades for best income configuration
if len(active) > 0:
    best_tf = best_income['tf']
    best_trail = best_income['trail']
    
    results_map = {"1H": results_1h, "4H": results_4h, "8H": results_8h}
    best_pf = results_map[best_tf][best_trail]["pf"]
    
    trades = best_pf.trades.records_readable
    
    print(f"\n{'='*100}")
    print(f"TRADES FOR BEST INCOME CONFIG: {best_tf} @ {best_trail*100:.0f}% trail")
    print(f"{'='*100}")
    
    # Format for display
    display_df = trades[["Entry Timestamp", "Exit Timestamp", "Return", "PnL"]].copy()
    display_df["Return"] = display_df["Return"] * 100
    display_df["Duration"] = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dt.days
    
    print(display_df.to_string())
    
    print(f"\nTotal trades: {len(trades)}")
    print(f"Winners: {(trades['PnL'] > 0).sum()} | Losers: {(trades['PnL'] < 0).sum()}")

## 9. Equity Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Plot best from each approach
configs_to_plot = [
    ("4H 30% (Swing)", results_4h.get(0.30)),
    ("4H 15% (Active)", results_4h.get(0.15)),
    ("4H 10% (Day)", results_4h.get(0.10)),
    ("8H 15%", results_8h.get(0.15)),
    ("1H 10%", results_1h.get(0.10)),
]

for name, result in configs_to_plot:
    if result:
        pf = result["pf"]
        m = result["metrics"]
        pf.value().resample('D').last().plot(ax=ax, label=f"{name}: {m['return']:+,.0f}% ({m['trades']} trades)")

ax.set_title("STRAT-002: Trail Stop Comparison")
ax.set_ylabel("Portfolio Value ($)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 10. Final Recommendation

In [ ]:
print("\n" + "="*100)
print("FINAL RECOMMENDATION")
print("="*100)

print("""
FOR SWING TRADING (Max Returns):
  → 4H or Daily @ 30% trail
  → ~2 trades/year, hold 6 months
  → Best total return

FOR ACTIVE INCOME (More Trades):
  → 4H @ 15% trail
  → ~8 trades/year, hold 6 weeks
  → Good balance of return vs frequency

FOR DAY TRADING (Max Frequency):
  → 4H @ 10% trail OR 1H @ 10% trail
  → More trades but lower win rate
  → Higher drawdowns, more volatility

KEY INSIGHT:
  On-chain signals (SOPR, RL z-score) are MACRO indicators.
  They identify rare capitulation events, not frequent trades.
  
  For true day trading with many trades, need different signals:
  - Price action / technical indicators
  - Order flow data
  - Funding rates
""")